# Bonus: a national microsimulation in free Colab, via sampling

The full Populace file (57,240 calibrated households, 343M weighted people) needs ~32+ GB of RAM to simulate — beyond free Colab's 12.7 GB. But the *file* is only 354 MB, so you can download it, take a **uniform household subsample**, scale the weights, and run the microsimulation on the sample — entirely inside free Colab.

Two tricks make it fit:
1. **Sample before simulating** — 5,000 households ≈ 1/11 of the compute and memory.
2. **Run each simulation in a subprocess** — baseline and reform back-to-back in one kernel exceeds 12.7 GB (the OS can't reclaim allocator fragmentation); a subprocess per pass caps the peak at a single simulation (~10–12 GB, measured).

**Accuracy at K=5,000 (uniform, seeded):** aggregates land within a few percent of the full file — this run: net income +3.7%, a \$3,000 CTC's national cost +4.7% (\$33.5B vs \$32.0B). Good for exploration and teaching; use the full file (32+ GB machine) or the web app for publication numbers.

Outputs below are from a real free-tier Colab run (July 2026). Runtime ≈ 5 minutes total.

In [1]:
%pip install -q "policyengine[us]" tables
print('ready')

ready


In [2]:
# V2: subprocess isolation per pass (OS reclaims memory fully between sims)
import subprocess, sys, time, textwrap
RUNNER = textwrap.dedent('''
    import sys, time, gc, threading
    import numpy as np, pandas as pd, psutil
    mode = sys.argv[1]
    proc=psutil.Process(); rss=lambda: proc.memory_info().rss/1e9
    peak=[0.0]
    def _poll():
        while True:
            peak[0]=max(peak[0],rss()); time.sleep(0.4)
    threading.Thread(target=_poll,daemon=True).start()
    from huggingface_hub import hf_hub_download
    P=hf_hub_download('policyengine/populace-us','populace_us_2024.h5',repo_type='dataset',revision='populace-us-2024-sparse-l0-refit-57k-71a0887-national-only-20260701')
    frames={k:pd.read_hdf(P,k) for k in ['person','household','tax_unit','spm_unit','family','marital_unit']}
    K=5000; rng=np.random.default_rng(42); hh=frames['household']; N=len(hh)
    keep=set(rng.choice(hh['household_id'].values,size=K,replace=False).tolist())
    person=frames['person'][frames['person']['person_household_id'].isin(keep)].copy()
    sub={'person':person,'household':hh[hh['household_id'].isin(keep)].copy(),
     'tax_unit':frames['tax_unit'][frames['tax_unit']['tax_unit_id'].isin(set(person['person_tax_unit_id']))].copy(),
     'spm_unit':frames['spm_unit'][frames['spm_unit']['spm_unit_id'].isin(set(person['person_spm_unit_id']))].copy(),
     'family':frames['family'][frames['family']['family_id'].isin(set(person['person_family_id']))].copy(),
     'marital_unit':frames['marital_unit'][frames['marital_unit']['marital_unit_id'].isin(set(person['person_marital_unit_id']))].copy()}
    scale=N/K
    for df in sub.values():
        for c in df.columns:
            if c.endswith('_weight'): df[c]=df[c]*scale
    del frames,person; gc.collect()
    from policyengine_us import Microsimulation
    from policyengine_core.simulations.simulation_builder import SimulationBuilder
    from policyengine_core.reforms import Reform
    reform=None
    if mode=='reform':
        reform=Reform.from_dict({'gov.irs.credits.ctc.amount.base[0].amount':{'2025-01-01.2025-12-31':3000}},country_id='us')
    ms=Microsimulation(reform=reform) if reform else Microsimulation()
    system=ms.tax_benefit_system
    b=SimulationBuilder(); b.populations=system.instantiate_entities()
    p=sub['person']
    b.declare_person_entity('person',p['person_id'].values)
    ents=[('household','person_household_id'),('spm_unit','person_spm_unit_id'),('family','person_family_id'),('tax_unit','person_tax_unit_id'),('marital_unit','person_marital_unit_id')]
    for ent,col in ents: b.declare_entity(ent,np.unique(p[col].values))
    for ent,col in ents: b.join_with_persons(b.populations[ent],p[col].values,np.array(['member']*len(p)))
    ms.build_from_populations(b.populations)
    idc={'person_id','household_id','person_household_id','spm_unit_id','person_spm_unit_id','family_id','person_family_id','tax_unit_id','person_tax_unit_id','marital_unit_id','person_marital_unit_id'}
    for ent in ['person','household','spm_unit','family','tax_unit','marital_unit']:
        df=sub[ent]
        for c in df.columns:
            if c not in idc and c in system.variables: ms.set_input(c,2024,df[c].values)
    v=float(ms.calculate('household_net_income',2025).sum())
    print(f'MODE={mode} NET={v:.6e} PEAK={peak[0]:.1f}',flush=True)
''')
open('/content/_pass.py','w').write(RUNNER)
def run_pass(mode):
    t0=time.time()
    r=subprocess.run([sys.executable,'/content/_pass.py',mode],capture_output=True,text=True)
    line=[l for l in r.stdout.splitlines() if l.startswith('MODE=')]
    if not line:
        print(f'{mode} FAILED rc={r.returncode}; tail: '+r.stdout[-200:]+r.stderr[-300:],flush=True); return None,None
    parts=dict(kv.split('=') for kv in line[0].split())
    print(f'[{mode}] {time.time()-t0:.0f}s net=${float(parts["NET"])/1e12:.3f}T childpeak={parts["PEAK"]}GB',flush=True)
    return float(parts['NET']),float(parts['PEAK'])
net,pk1=run_pass('baseline')
ref,pk2=run_pass('reform')
if net and ref:
    print(f'RESULT-V2 cost=${(ref-net)/1e9:.1f}B (truth 32.0B) | peaks {pk1}/{pk2}GB of 12.7GB',flush=True)


[baseline] 108s net=$15.041T childpeak=10.2GB
[reform] 122s net=$15.075T childpeak=11.8GB
RESULT-V2 cost=$33.5B (truth 32.0B) | peaks 10.2/11.8GB of 12.7GB


## Notes
- `K` controls the trade-off: smaller = less RAM and time, noisier estimates. K=5,000 peaked at 11.8 GB of 12.7 GB — if your VM is tight, drop to K=4,000.
- The sample is drawn from **Populace** ([github.com/PolicyEngine/populace](https://github.com/PolicyEngine/populace)), PolicyEngine's open, calibrated US microdata.
- A small pre-calibrated national file that makes this unnecessary is tracked in [populace#213](https://github.com/PolicyEngine/populace/issues/213).
- Questions: max@policyengine.org